In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import glob
import cv2
from scipy import stats

from skimage.measure import shannon_entropy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Dataset Analysis and Signal Processing: SETI Radio Spectrograms

## Abstract
In the search for extraterrestrial intelligence (SETI), raw radio signals captured by telescopes (such as the Allen Telescope Array) are converted into two-dimensional spectrograms (Time vs. Frequency). This exploratory data analysis (EDA) examines the structural, statistical, and spatial properties of a 7,000-image simulated SETI dataset from Kaggle. Through signal processing, color-space mapping transformations, distribution analysis, and hypothesis testing, we uncover the physical characteristics of these technosignatures.

In [ ]:
# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale = 1.1)

In [ ]:
# Define dataset paths
DATA_DIR = "../data/raw"
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VALID_DIR = os.path.join(DATA_DIR, 'valid')
TEST_DIR = os.path.join(DATA_DIR, 'test')

In [ ]:
CLASSES = sorted([c for c in os.listdir(TRAIN_DIR) if not c.startswith('.')])
print(f"Environment initialized. Target classes detected: {CLASSES}")

## 1. Basic Dataset Expectations & Shape Verification
Before any modeling, we must verify the structural dimensions of our data. We are going to count how many images each split  has (Train, Validation, Test), verify class balance, and inspect the spatial dimensions (resolution and channel shape) of individual spectrogram images.

In [ ]:
def test_dataset_shapes_and_splits():
    # 1. Count splits
    splits = {'train': TRAIN_DIR, 'valid': VALID_DIR, 'test': TEST_DIR}
    split_totals = {}
    for split_name, split_path in splits.items():
        total = len(glob.glob(os.path.join(split_path, '**', '*.png'), recursive = True))
        split_totals[split_name] = total
        print(f"{split_name.lower()}: {total} images")
    print(f"Total Dataset Size: {sum(split_totals.values())} images\n")

    # 2. Check class balance & image shapes in Train set
    class_stats = []
    sample_shape = None
    
    for cls in CLASSES:
        cls_path = os.path.join(TRAIN_DIR, cls)
        img_paths = glob.glob(os.path.join(cls_path, '*.png'))
        count = len(img_paths)
        
        if count > 0:
            # Read sample image to check shape
            sample_img = cv2.imread(img_paths[0], cv2.IMREAD_UNCHANGED)
            sample_shape = sample_img.shape
            
        class_stats.append({'Class': cls, 'Count': count})

    df_classes = pd.DataFrame(class_stats)
    print("=== Training Class Balance ===")
    print(df_classes.to_string(index=False))
    print(f"\nNative Image Tensor Shape: {sample_shape} (Height, Width, Channels)")
    
    return df_classes

df_class_distribution = test_dataset_shapes_and_splits()

The total dataset consists of **7,000 spectrogram images**, cleanly partitioned into **5,600 training samples (80%)**, **700 validation samples (10%)**, and **700 test samples (10%)**.

Every single one of the 7 technosignature classes (`brightpixel`, `narrowband`, `narrowbanddrd`, `noise`, `squarepulsednarrowband`, `squiggle`, and `squigglesquarepulsednarrowband`) contains **800 training images**. The dataset doesn’t have class imbalance.

The native image dimensions are **$384 \times 512$ pixels with 4 channels** `(384, 512, 4)`. The 4-channel configuration (typically RGBA or multi-layered data feeds common in radio astronomy waterfall plots) poses a technical constraint. Standard pre-trained Computer Vision backbones (such as ResNet50 or EfficientNet) natively accept **3-channel RGB tensors**. Therefore, our data pipeline must handle channel mapping or reduction (dropping the alpha channel or converting to a 3-channel layout) during tensor transformation. Furthermore, because native dimensions ($384 \times 512$) do not match standard CNN input requirements ($224 \times 224$), spatial resizing transformations are mandatory.

## 2. Domain Knowledge: What Each Class Represents
These classes, based on SETI Institute definitions, represent the following:

In [ ]:
def plot_raw_class_samples():
    fig, axes = plt.subplots(1, 7, figsize = (22, 4))
    fig.suptitle('Raw Spectrogram Samples Across All 7 SETI Classes', fontsize = 18, y = 1.05)
    
    for i, cls in enumerate(CLASSES):
        img_path = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[0]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        ax = axes[i]
        ax.imshow(img, cmap = 'gray')
        ax.set_title(cls, fontsize = 10, pad = 8)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

plot_raw_class_samples()

1. **Noise:** Pure cosmic background static. It lacks any concentrated structure.
2. **Brightpixel:** A sudden, highly intense burst of energy confined to a tiny frequency band and time window.
3. **Narrowband:** A continuous signal at a single, unchanging frequency. It appears as a straight, unbroken line.
4. **Narrowbanddrd:** A narrowband signal with **Doppler Drift**. The line is diagonal, representing the relative acceleration between Earth and the extraterrestrial transmitter.
5. **Square Pulsed Narrowband:** A narrowband signal that turns on and off at regular intervals (like a metronome or Morse code), appearing as a dashed line.
6. **Squiggle:** A signal whose frequency drifts in a wavy, sinusoidal pattern, possibly due to a rotating transmitter or complex modulation.
7. **Squigglesquarepulsednarrowband:** A wavy, drifting signal that is also pulsing on and off.

## 3. Color-Space Transformations and Information Hiding
Human eyes are bad at distinguishing subtle variations in monochrome (grayscale) intensity, but highly sensitive to color gradients. 

We are going to take a single signal (`squiggle`) and render it using different matplotlib colormaps (`gray`, `magma`, `viridis`, `jet`). Grayscale images can "hide" low-amplitude signal tracks within background noise because the dynamic range is compressed for human viewing. Colormaps like `magma` or `viridis` highlight hidden frequency structures.

In [ ]:
def test_color_space_transformations():
    # A sample squiggle image
    sample_path = glob.glob(os.path.join(TRAIN_DIR, 'squiggle', '*.png'))[7]
    img = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
    
    colormaps = ['gray', 'magma', 'viridis', 'jet', 'inferno']
    
    fig, axes = plt.subplots(1, len(colormaps), figsize = (20, 4))
    fig.suptitle('Perceptual Impact of Color Mapping on Signal Visibility (Squiggle Class)', fontsize = 16, y = 1.05)
    
    for i, cmap in enumerate(colormaps):
        ax = axes[i]
        im = ax.imshow(img, cmap = cmap)
        ax.set_title(f"Colormap: {cmap}", fontsize = 11)
        ax.axis('off')
        fig.colorbar(im, ax = ax, orientation = 'horizontal', fraction = 0.046, pad = 0.04)
        
    plt.tight_layout()
    plt.show()

test_color_space_transformations()

In the **gray** rendering, the faint, curving trajectory of the squiggle signal can easily blur into the background space noise because the human eye has a limited dynamic range for distinguishing shades of gray.

Perceptually uniform colormaps like **magma**, **viridis**, and **inferno** smoothly expand the color spectrum. They map low-intensity background noise to deep, dark tones (purples/blacks) and high-intensity technosignature tracks to glowing, high-contrast hues (yellows/oranges). This enhances the visual distinction of the signal's spatial curvature.

The **jet colormap** provides sharp color transitions (from blue to green to red), which can make high-amplitude peaks stand out instantly, though it is non-uniform and can sometimes introduce false visual boundaries.

## 4. Global Pixel Intensity & Dynamic Range Distribution
We are going to compute pixel intensity histograms for each class across the 0-255 range. This will reveal the underlying distribution of the background static versus signal power. A narrow peak near 0 indicates a vacuum/quiet space background, while heavy right-tails represents high-power technosignatures.

In [ ]:
def test_pixel_intensity_distributions():
    plt.figure(figsize = (14, 6))
    
    for cls in CLASSES:
        paths = glob.glob(os.path.join(TRAIN_DIR, cls, '*.png'))[:50] # Sample 50 images per class
        pixels = []
        for p in paths:
            img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
            pixels.extend(img.flatten())
            
        sns.kdeplot(pixels, label = cls, fill = False, linewidth = 2)
        
    plt.title('Pixel Intensity Probability Density Functions (PDF) per Class', fontsize = 16)
    plt.xlabel('Pixel Intensity Value (0 - 255)')
    plt.ylabel('Density')
    plt.xlim(0, 255)
    plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
    plt.tight_layout()
    plt.show()

test_pixel_intensity_distributions()

All seven distribution curves exhibit a sharp, towering peak clustered toward the lower end of the intensity spectrum (near $0$). This represents the vacuum/background space static. Because the vast majority of pixels in any given spectrogram represent empty space or ambient noise rather than signal tracks, the background overwhelmingly dictates the global pixel distribution.

While the curves follow a similar baseline path due to the shared background noise, signal-bearing classes (such as `narrowband`, `squiggle`, and `squarepulsednarrowband`) show gradual "tails" extending toward higher intensity values (bright regions).The noise class drops off more steeply because it lacks high-amplitude signal paths.